In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn import linear_model
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE

# statsmodels
import statsmodels.api as sm
from statsmodels.api import qqplot
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
import scipy.stats as stats


In [ ]:
import urllib.request
import zipfile
import os

# 1. Download the official zip file from the UCI academic repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip"
urllib.request.urlretrieve(url, "bike.zip")

# 2. Extract ONLY the day.csv file we need
with zipfile.ZipFile("bike.zip", "r") as zip_ref:
    zip_ref.extract("day.csv")

# 3. Clean up and delete the zip file so your folder stays pristine
os.remove("bike.zip")
print("day.csv successfully downloaded!")

In [ ]:
df = pd.read_csv('day.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
#drop instant variable this is just an index and doesn't add to the prediction
df = df.drop(columns=['instant'])

- cast dteday to datetime 

In [ ]:
# convert dteday to datetime 
df['dteday'] = pd.to_datetime(df['dteday'])

In [ ]:
sns.pairplot(df)

In [ ]:
df.columns

# Outlier Detection

- While taking a glance at the data, there seem to be outliers in windspeed these values don't vary by a large margin , this is mainly because there is not much variance in windspeed , i.e that scale is not large

In [ ]:
fig,ax = plt.subplots(1,4,figsize=(15,5))
for idx,i in enumerate(['temp','atemp','hum','windspeed']) : 
    sns.boxplot(x=df[i],ax=ax[idx])
    ax[idx].set_title(i)
plt.show()

In [ ]:
fig,ax = plt.subplots(1,4,figsize=(15,5))
for idx,i in enumerate(['temp','atemp','hum','windspeed']) : 
    sns.histplot(x=df[i],ax=ax[idx],kde=True)
    ax[idx].set_title(i)
plt.show()

In [ ]:
fig,ax = plt.subplots(1,4,figsize=(15,5))
for idx,i in enumerate(['temp','atemp','hum','windspeed']) : 
    sns.scatterplot(x=df[i],y=df['cnt'],ax=ax[idx])
    ax[idx].set_title(i)
plt.show()

- temp and atemp seem to affect cnt in the same way which is expected
- Similar behavior can be seen with humidity and windspeed

In [ ]:
sns.regplot(x=df['hum'],y=df['windspeed'])

- Create feature Temperature Humidity Index which captures both humidity and temperature in a single feature
- THW = HI - (1.072 * W)

In [ ]:
df['thw'] = df['hum'] - (1.072 * df['windspeed'])

- Though the column thw was derived it can be observed from the corr matrix that there is very poor relationship between thw and cnt and so is the case with hum 
- drop both hum and thw

In [ ]:
corr_mat = df[['thw','hum','windspeed','cnt']]
plt.figure(figsize=(25,15))
corr_matrix = corr_mat.corr().where(np.triu(np.ones(corr_mat.corr().shape),k=1).astype('bool'))
sns.heatmap(corr_matrix,annot=True)
plt.show()

In [ ]:
# drop columns humidity and thw
df = df.drop(['hum','thw'],axis=1)

- The seasons column seems to be represented as an ordinal variable
- There is no clear order to the seasons column , hence this shouldn't be ordinal
- Since regression model doesn't take categorical variables ,dummy encode these variables 
- Point to be noted here is that when applying the dummy encoding it is a good practice to take n-1 features when there are n     variables . This is because (n-1) features are enough to explain all possible scenarios , making the nth feature highly     correlated to the other 1,..,n-1 features. 
- Multi Collinearity can confuse the model ,by making it too complex , another way to interpret it would be that in linear regression coefficient m1 explains the behavior of y as x1 varies , while all other parameters are held constant . This fundamental interpretability of linear regression doesn't hold 

In [ ]:
# The seasons variable contains 4 values
df['season'].unique()

In [ ]:
convert_to_categorical = {1:'spring', 2:'summer', 3:'fall', 4:'winter'}
df['season'] = df['season'].replace(convert_to_categorical)
df['season'].unique()

In [ ]:

sns.catplot(data=df,x='season',y='cnt')


In [ ]:
sns.catplot(data=df,x='season',y='cnt',estimator='mean',kind='point')

In [ ]:
fig,ax = plt.subplots(1,4,figsize=(15,5))
for idx,(sidx,seasons) in enumerate(df.groupby(['season'])) : 
    sns.histplot(data=seasons,x='cnt',kde=True,ax=ax[idx])
    ax[idx].set_title(sidx)
plt.show()

In [ ]:
sns.displot(data=df,x='cnt',hue='season',kind='kde')

- There seems to be positive linear relationship in the order spring->winter->summer->fall

In [ ]:
sns.boxplot(data=df,y='cnt',x='season')
plt.show()

In [ ]:
# The weathersits variable contains 4 values
df['weathersit'].unique()

In [ ]:
convert_to_categorical = {i : 'weathersit_' + str(i)  for i in df['weathersit'].unique()}
df['weathersit'] = df['weathersit'].replace(convert_to_categorical)
df['weathersit'].unique()

In [ ]:

sns.catplot(data=df,x='weathersit',y='cnt',estimator='median')


In [ ]:
sns.catplot(data=df,x='weathersit',y='cnt',estimator='median',kind='point')

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(15,5))
for idx,(sidx,weather) in enumerate(df.groupby(['weathersit'])) : 
    sns.histplot(data=weather,x='cnt',kde=True,ax=ax[idx])
    ax[idx].set_title(sidx)
plt.show()

In [ ]:
sns.displot(data=df,x='cnt',hue='weathersit',kind='kde')

- here too here seems to be a positive linear correlation in the order of weather3->weahter-2->weather-1

In [ ]:
sns.boxplot(data=df,y='cnt',x='weathersit')
plt.show()

In [ ]:
num_to_week_day = {1 : 'monday',2 : 'tuesday',3 : 'wednesday',4 : 'thursday',5 : 'friday',6:'saturday',0:'sunday' }
df['weekday']=df['weekday'].replace(num_to_week_day)

In [ ]:
df['weekday'].value_counts()

In [ ]:
df['weekday'] = df['weekday'].astype('category').cat.set_categories(['sunday','monday','tuesday','wednesday','thursday','friday','saturday'],ordered=True)

In [ ]:

sns.catplot(data=df,x='weekday',y='cnt',estimator='mean')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# order_weekday = df.groupby(['weekday'])['cnt'].mean().sort_values().index
sns.catplot(data=df,x='weekday',y='cnt',estimator='mean',kind='point',order=None)
plt.xticks(rotation=90)
plt.show()

In [ ]:
xm = 2
ym = 4
fig,ax = plt.subplots(xm,ym,figsize=(15,10))
# ax.axis('off')
# ax = ax.reshape(-1,1)
for idx,(sidx,week) in enumerate(df.groupby(['weekday'])) :
    gc = idx % ym
    gr = idx // ym
    sns.histplot(week['cnt'],kde=True,ax=ax[gr,gc])
    ax[gr,gc].set_title(sidx)
[fig.delaxes(i) for i in ax.flat if not i.has_data()]
plt.tight_layout()
plt.show()

In [ ]:
sns.displot(data=df,x='cnt',hue='weekday',kind='kde')

In [ ]:
sns.boxplot(data=df,y='cnt',x='weekday',order = None)
plt.xticks(rotation=45)
plt.show()

In [ ]:
df['weekday']

In [ ]:
df[df['workingday'] == 0][df['holiday']== 0]['weekday'].unique()

- Let's say today is a working day , in that case holiday = 0 , workingday =1 and weekday has one of the 7 days
- Today is a holiday , in that case holiday = 1 , workingday = 0 , weekday has one of the 7 days
- Today is a weekend , in that case holiday = 0 , workingday = 0 , weekday has to be saturday or sunday

In [ ]:
df['mnth'] = df['dteday'].dt.month_name()

In [ ]:

sns.catplot(data=df,x='mnth',y='cnt',estimator='mean')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# order_month = df.groupby(['mnth'])['cnt'].mean().sort_values().index
sns.catplot(data=df,x='mnth',y='cnt',estimator='mean',kind='point',order=None)
plt.xticks(rotation=90)
plt.show()

In [ ]:
df['mnth'] = df['mnth'].astype("category").cat.set_categories(df['mnth'].unique(),ordered=True)

In [ ]:
xm = 3
ym = 4
fig,ax = plt.subplots(xm,ym,figsize=(15,10))
# ax.axis('off')
# ax = ax.reshape(-1,1)
for idx,(sidx,month) in enumerate(df.groupby(['mnth'])) :
    gc = idx % ym
    gr = idx // ym
    sns.histplot(month['cnt'],kde=True,ax=ax[gr,gc])
    ax[gr,gc].set_title(sidx)
[fig.delaxes(i) for i in ax.flat if not i.has_data()]
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=df,y='cnt',x='mnth',order = None)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Dummy encoding categorical variables
df = pd.concat([df,pd.get_dummies(df['season'],drop_first=True)],axis=1)
df = pd.concat([df,pd.get_dummies(df['weathersit'],drop_first=True)],axis=1)
df = pd.concat([df,pd.get_dummies(df['weekday'],drop_first=True)],axis=1)
df = pd.concat([df,pd.get_dummies(df['mnth'],drop_first=True)],axis=1)

In [ ]:
# drop the seasons column as all information in the feature has been captured
df = df.drop(['season','weathersit','weekday','mnth'],axis=1)

In [ ]:
#dteday spans over 2 years
df['dteday'].dt.year.unique()

In [ ]:
df['yr'].value_counts()

- It is clear that 2018 is being represented by 0 and 2019 by 1 
- convert year to dummy variable 
- it can also be observed that the no.of rentals are increasing over the years


In [ ]:
sns.pointplot(x=df['yr'].replace({0:2018,1:2019}),y=df['cnt'])

In [ ]:
df = df.drop(columns = 'dteday')


In [ ]:
df.columns

In [ ]:
# Remove the columns casual and registered because their sum is basically the target column , such columns are not usually present in real time predictions as these are target data
df = df.drop(columns = ['casual','registered'])

In [ ]:
re_order_columns = list(df.columns[~df.columns.str.contains('cnt')]) + ['cnt']
re_order_columns


In [ ]:
df = df[re_order_columns]

In [ ]:
df.shape

## Target Variable Distribution

Before modeling, we check whether `cnt` (total rentals) is normally distributed. Linear regression performs better when the target is not heavily skewed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original scale
sns.histplot(df['cnt'], kde=True, ax=axes[0])
axes[0].set_title('cnt — Original Scale')
axes[0].set_xlabel('cnt')

# Log scale
sns.histplot(np.log1p(df['cnt']), kde=True, ax=axes[1])
axes[1].set_title('log1p(cnt) — Log Scale')
axes[1].set_xlabel('log1p(cnt)')

plt.tight_layout()
plt.show()

print(f'Skewness (original): {df["cnt"].skew():.3f}')
print(f'Skewness (log):      {np.log1p(df["cnt"]).skew():.3f}')


The log transform reduces skewness significantly. We will train a parallel model on `log1p(cnt)` and compare results at the end.

- It can be inferred from the heatmap that yr,temp,atemp have a high positive correlation betweem yr,temp and atemp
- It can also be seen that there is a very high positive correlation  between temp and atemp 
- Collinearity causes problems in the way linear regression works (causes issues with interpretability)
- Hence either temp or atemp must be removed

In [ ]:
plt.figure(figsize=(25,15))
corr_matrix = df.corr().where(np.triu(np.ones(df.corr().shape),k=1).astype('bool'))
sns.heatmap(corr_matrix,annot=True)
plt.show()

In [ ]:
# Saturday seems to have a high correlation with workingdays , this can lead multi-collinearity hence remove
df = df.drop(columns='saturday')

In [ ]:
# Filter out dummy encoded values as it doesn't make sense to measure correlation on these variables
corr_mat = df[[ 'temp', 'atemp',
       'windspeed','cnt']]

In [ ]:
# Remove the lower triangle and the diagonal of the correlation matrix as it is symetric 
plt.figure(figsize=(15,5))
corr_matrix = corr_mat.corr().where(np.triu(np.ones(corr_mat.corr().shape),k=1).astype('bool'))
sns.heatmap(corr_matrix,annot=True)
plt.show()

In [ ]:
# dropping atemp because it is highly correlated 
df = df.drop(columns = 'atemp')

In [ ]:
plt.figure(figsize=(15,5))
corr_mat= corr_mat.drop(columns = 'atemp')
corr_matrix = corr_mat.corr().where(np.triu(np.ones(corr_mat.corr().shape),k=1).astype('bool'))
sns.heatmap(corr_matrix,annot=True)
plt.show()

In [ ]:
fig,ax = plt.subplots(1,2,figsize=(15,5))
for idx,i in enumerate(['temp', 'windspeed']) : 
    sns.scatterplot(data=df,x=i,y='cnt',ax=ax[idx])
    ax[idx].set_title('{} Vs cnt'.format(i))
plt.show()

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(15,5))
for idx,i in enumerate(['yr', 'workingday','holiday']) : 
    sns.pointplot(data=df,x=i,y='cnt',ax=ax[idx])
    ax[idx].set_title('{} Vs cnt'.format(i))
plt.show()

In [ ]:
# Altohugh here a shadow of a linear relationship can be observed this isn't clearly visible as there is a good degree of variance
sns.pairplot(df[['temp', 'windspeed']])

- Minmax scaler was used here 
- The scaler handles outliers if any and brings data into a scale of 0 to 1 
- This prevents overfitting , i.e giving too much weights to certain features
- In this case , though data is pretty much on a similar scale , so adding scaler shouldn't make too much difference

In [ ]:
def get_model_ready(df) : 
    df1 = df.copy()
    y = df1.pop("cnt")
    X = df1
    X_train,X_test,y_train,y_test  = train_test_split(X,y,train_size=0.7, random_state=100)
    cols = X_train.columns
    scaler = MinMaxScaler()
    num_features = ['temp','windspeed']
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    X_train2 = scaler.fit_transform(X_train[num_features])
    X_test2 = scaler.transform(X_test[num_features])
    X_train2 = pd.DataFrame(X_train2,columns=num_features)
    X_test2 = pd.DataFrame(X_test2,columns=num_features)
    X_train = X_train.drop(columns = num_features)
    X_test = X_test.drop(columns = num_features)
    X_train = pd.concat([X_train,X_train2],axis=1)
    X_test = pd.concat([X_test,X_test2],axis=1)
    y_train = y_train.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)
    return X_train,X_test,y_train,y_test

In [ ]:
X_train,X_test,y_train,y_test = get_model_ready(df)

In [ ]:
X_train

In [ ]:
print(f'num_of features : {X_train.shape[1]}')

#  Feature Selection I

   # Recursive Feature Elimination

In [ ]:
def rfe_method(X_train,y_train,n) : 
    estimator  = linear_model.LinearRegression()
    rfe_feature_selection  = RFE(estimator, n_features_to_select=n, step=1)
    rfe_feature_selection = rfe_feature_selection.fit(X_train,y_train)
    rankings = rfe_feature_selection.ranking_
    support  = rfe_feature_selection.support_
    rfe_df = pd.DataFrame(data= {'features' : X_train.columns,'support' : support,'ranking' : rankings,})
    return rfe_df.sort_values(['ranking']),support

# Training using Stats Model

In [ ]:
def stats_model_training(X_train,y_train) : 
    X_train_rfe = sm.add_constant(X_train)
    lm = sm.OLS(y_train,X_train_rfe).fit()
    return lm.summary()

In [ ]:
def vif_method(X) : 
    vif = pd.DataFrame()
    vif['Features'] = X.columns
    vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif['VIF'] = round(vif['VIF'], 2)
    vif = vif.sort_values(by = "VIF", ascending = False)
    return vif

In [ ]:
rfe_df,support = rfe_method(X_train.copy(),y_train.copy(),16)
rfe_df

In [ ]:
X_train = X_train[X_train.columns[support]]
X_test = X_test[X_test.columns[support]]

- If p(t) > 0.05 then that means we fail to reject the null hypothesis , hence the feature is insignificant remove it 
- if vif > 5 then there might be high multicollinearity , if vif > 10 then there is high multicollinearity for sure
- if vif < 5 then there is very low to no multicollinearity
- p(f-statistic) > 0.05 then that means we fail to reject the null hypothesis , hence the variance in the model isn't statistically significant

- In cases where there is high p-value and high vif-value for different features, prioritize taking care of the feature with high p-value , as vif can change as the feature set changes 

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
# In the corr matrix it could be seen that there was a -0.25 correlation between holiday and workingday , since these cover all weekdays removing that should improve the relationship
X_train = X_train.drop(columns = 'holiday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'workingday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'monday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'tuesday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'thursday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'wednesday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'friday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

# Residual Analysis

In [ ]:
X_train_rfe = sm.add_constant(X_train)
lm = sm.OLS(y_train,X_train_rfe).fit()

In [ ]:
y_pred_train = lm.predict(X_train_rfe)

In [ ]:
(y_pred_train - y_train).mean()

- It can be seen that mean of the residuals is centered around zero(infestimally close to zero) which is in-line with the assumptions of linear regression

In [ ]:
sns.displot(x=(y_pred_train - y_train),kde=True,stat='probability')
plt.xlabel('residuals')
plt.axvline(0,color='r')
plt.show()

- As seen except for a few points the variance of the model seems to be constant to a certain extent 
- It might be that by increasing the data or collecting data for a new feature might increase this 
- Also it might be possible to derive other features from the existing data 

In [ ]:
sns.relplot(x=y_pred_train,y=y_train,kind='scatter')
plt.title('y_pred Vs y_train')
plt.xlabel('y_pred')
plt.ylabel('y_train')
plt.show()

In [ ]:
res  = (y_pred_train - y_train)
sns.scatterplot(x=y_train,y=res)
plt.axhline(res.mean(),color='red')
plt.ylabel('residuals')
plt.show()

- It can be inferred from the qq-plot that the residuals almost follow a normal distribution 
- It can be observed that the points are tappering off at the right tail

In [ ]:
def plot_qq(res) : 
    qqplot(res,line='45',fit=True,dist=stats.norm)
    plt.show()


In [ ]:
plot_qq(res)

In [ ]:
X_test = X_test[X_train.columns]

### Durbin-Watson Test

Checks for autocorrelation in residuals. A value close to 2 indicates no autocorrelation, <1 or >3 indicates a problem.

In [ ]:
dw_stat = durbin_watson(y_pred_train - y_train)
print(f'Durbin-Watson statistic: {dw_stat:.4f}')
if 1.5 < dw_stat < 2.5:
    print('✓  No significant autocorrelation detected (value near 2.0)')
else:
    print('⚠  Possible autocorrelation — consider time-aware modeling')


### RMSE on Training Set

In [ ]:
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f'Train RMSE: {rmse_train:.2f}')


# The r2-score for RFE is 0.77

In [ ]:
X_test_lm = sm.add_constant(X_test)
y_pred = lm.predict(X_test_lm)
round(r2_score(y_pred,y_test),2)

# Deriving Model Metrics using sklearn

In [ ]:
lr = linear_model.LinearRegression()

In [ ]:
lr.fit(X_train,y_train)

# Model Evaluation

- The coefficients and the intercept of the sklearn linear regression model match that of the 

In [ ]:
lr.coef_

In [ ]:
lr.intercept_

In [ ]:
X_test = X_test[X_train.columns]

In [ ]:
y_pred = lr.predict(X_test)

In [ ]:
print(f"R2 Score of model - {round(r2_score(y_pred,y_test),2)}")

In [ ]:
sns.relplot(x=y_pred,y=y_test,kind='scatter')
plt.title('y_pred Vs y_test')
plt.xlabel('y_pred')
plt.ylabel('y_test')
plt.show()

In [ ]:
abs(pd.Series(lr.coef_,X_train.columns)).sort_values()

In [ ]:

coef_vec = [[coef,cols ] for coef,cols in zip(lr.coef_,X_train.columns)] 


In [ ]:
pd.Series([str(coef) + ' * ' + col + ' + ' for coef,col in coef_vec ]).sum().replace('+ -','-').replace('-','- ')

- It can be inferred that temperature , weathersit_3 and yr have contributued the most towards prediction 

In [ ]:
pd.Series(abs(lr.coef_),X_train.columns).sort_values().plot(kind='bar')

In [ ]:
rfe_cols = pd.Series(abs(lr.coef_),X_train.columns).index

# Feature Selection II

# Backward Feature Selection

- Backward feature selection is done by repeatedly training the model by interatively changing feature set
- In each iteration(epoch) , each feature is removed and the r2-score of the model is calculated . Then at the end of the iteration(epoch) ,the feature with the least importance (i.e the model r2-score remains high even in the abensence of the feature) is removed . 
- Then the feature set that remains undergoes the same process 
- This happens until the num_of features n(num_of features to select)
- In this way the n most important features are selected

In [ ]:
def train_model(X_train,y_train,X_test,y_test) : 
    estimator  = linear_model.LinearRegression()
    estimator.fit(X_train,y_train)
    y_pred = estimator.predict(X_test)
    r2  =  r2_score(y_pred,y_test)
    return 1 - ((1-r2) * (X_train.shape[0]-1) / (X_train.shape[0] - 1 - X_train.shape[1]))
def backward_method(X_train,y_train,X_test,y_test,n) : 
    while(len(X_train.columns) > n) : 
        r2_scores  = []
        cols = X_train.columns
        for i in cols : 
            X_trainc = X_train.copy()
            X_testc = X_test.copy()
            X_trainc = X_train[X_trainc.columns[~X_trainc.columns.str.contains(i)]]
            X_testc = X_testc[X_testc.columns[~X_testc.columns.str.contains(i)]]
            r2_scores.append(train_model(X_trainc,y_train,X_testc,y_test))
        idx = r2_scores.index(max(r2_scores))
        X_train = X_train.drop(columns =cols[idx])
        X_test = X_test.drop(columns =cols[idx])
    return X_train,X_test
    

In [ ]:
X_train,X_test,y_train,y_test = get_model_ready(df)
X_train,X_test = backward_method(X_train.copy(),y_train.copy(),X_test.copy(),y_test.copy(),16)

In [ ]:
X_train.columns

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'tuesday')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'May')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'March')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:

X_train = X_train.drop(columns = 'August')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train_rfe = sm.add_constant(X_train)
lm = sm.OLS(y_train,X_train_rfe).fit()

In [ ]:
y_pred_train = lm.predict(X_train_rfe)

 # The results of the model evaluations of RFE and Backward Feature Selection are almost the same 

In [ ]:
(y_pred_train - y_train).mean()

In [ ]:
sns.displot(x=(y_pred_train - y_train),kde=True)
plt.xlabel('residuals')
plt.axvline(0,color='r')
plt.show()

In [ ]:
sns.relplot(x=y_pred_train,y=y_train,kind='scatter')
plt.title('y_pred Vs y_train')
plt.xlabel('y_pred')
plt.ylabel('y_train')
plt.show()

In [ ]:
res  = (y_pred_train - y_train)
sns.scatterplot(x=y_train,y=res)
plt.ylabel('residuals')
plt.axhline(res.mean(),color='red')

In [ ]:
plot_qq(res)

In [ ]:
X_test = X_test[X_train.columns]

In [ ]:
X_test_lm = sm.add_constant(X_test)
y_pred = lm.predict(X_test_lm)
round(r2_score(y_pred,y_test),2)

In [ ]:
lr = linear_model.LinearRegression()

In [ ]:
lr.fit(X_train,y_train)

In [ ]:
lr.coef_

In [ ]:
lr.intercept_

In [ ]:
X_test = X_test[X_train.columns]

In [ ]:
y_pred = lr.predict(X_test)

# The r2-score for backward feature selection is 0.78

In [ ]:
print(f"R2 Score of model - {round(r2_score(y_pred,y_test),2)}")

In [ ]:
sns.relplot(x=y_pred,y=y_test,kind='scatter')
plt.title('y_pred Vs y_test')
plt.xlabel('y_pred')
plt.ylabel('y_test')
plt.show()

In [ ]:
abs(pd.Series(lr.coef_,X_train.columns)).sort_values()

In [ ]:

coef_vec = [[coef,cols ] for coef,cols in zip(lr.coef_,X_train.columns)] 

In [ ]:
pd.Series([str(coef) + ' * ' + col + ' + ' for coef,col in coef_vec ]).sum().replace('+ -','-').replace('-','- ')

In [ ]:
pd.Series(abs(lr.coef_),X_train.columns).sort_values().plot(kind='bar')

In [ ]:
bcf_cols  = pd.Series(abs(lr.coef_),X_train.columns).index

# Intersection of features of rfe model and bcf model

In [ ]:
np.intersect1d(rfe_cols,bcf_cols)

- Note that the the top 3 features of both models are similar i.e temp , weathersit_3 and yr
- This gives more confidence in the results of the model
- It can also be observed that the residual analysis represents the same insights in both the feature selections
- This gives us more confidence that the model is going in the right direction

### This tells us the fact that temp , yr and weathersit_3 are the 3 most import features to the model

- Both models trained after using different feature selection techniques gave similar results in terms of model evaluation and test

# Feature Ablation Study 

 - It can be observed that removing year column lead to a very sharp drop in R-square

In [ ]:
X_train,X_test,y_train,y_test = get_model_ready(df)

In [ ]:
X_train,X_test = backward_method(X_train.copy(),y_train.copy(),X_test.copy(),y_test.copy(),16)

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
print(vif_method(X_train.copy()))

In [ ]:
X_train = X_train.drop(columns = 'yr')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
X_train,X_test,y_train,y_test = get_model_ready(df)

In [ ]:
X_train,X_test = backward_method(X_train.copy(),y_train.copy(),X_test.copy(),y_test.copy(),16)

In [ ]:
X_train = X_train.drop(columns = 'temp')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

In [ ]:
X_train,X_test,y_train,y_test = get_model_ready(df)
X_train,X_test = backward_method(X_train.copy(),y_train.copy(),X_test.copy(),y_test.copy(),16)

In [ ]:
X_train = X_train.drop(columns = 'weathersit_3')

In [ ]:
print(stats_model_training(X_train.copy(),y_train.copy()))

---
# Extended Model Evaluation

The sections below go beyond the original single train/test split to provide more robust, comparable results: 5-fold cross-validation, a log-transform experiment, and benchmarking against regularized and ensemble models.

## 5-Fold Cross-Validation

A single 70/30 split gives one lucky (or unlucky) score. 5-fold CV trains and validates on 5 different subsets and reports mean ± std, which is a much more reliable estimate of generalisation performance.

In [ ]:
# Rebuild a clean feature set for CV
X_train_cv, X_test_cv, y_train_cv, y_test_cv = get_model_ready(df)

# Use the same feature columns selected by RFE above
# (re-run RFE to get support on the full feature set)
rfe_cv = RFE(linear_model.LinearRegression(), n_features_to_select=16, step=1)
rfe_cv.fit(X_train_cv, y_train_cv)
X_cv = X_train_cv[X_train_cv.columns[rfe_cv.support_]]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = linear_model.LinearRegression()

cv_r2    = cross_val_score(lr_cv, X_cv, y_train_cv, cv=kf, scoring='r2')
cv_rmse  = np.sqrt(-cross_val_score(lr_cv, X_cv, y_train_cv, cv=kf,
                                     scoring='neg_mean_squared_error'))

print('=== 5-Fold Cross-Validation — Linear Regression ===')
print(f'R²   per fold : {np.round(cv_r2, 3)}')
print(f'R²   mean ± std : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
print()
print(f'RMSE per fold : {np.round(cv_rmse, 1)}')
print(f'RMSE mean ± std : {cv_rmse.mean():.1f} ± {cv_rmse.std():.1f}')


## Log-Transform Experiment

We saw earlier that `cnt` has moderate right skew. Training on `log1p(cnt)` and converting predictions back with `expm1` often improves linear model performance on skewed targets.

In [ ]:
# Train on log-transformed target
X_train_cv, X_test_cv, y_train_cv, y_test_cv = get_model_ready(df)

# Apply log1p to target
y_train_log = np.log1p(y_train_cv)
y_test_log  = np.log1p(y_test_cv)

rfe_log = RFE(linear_model.LinearRegression(), n_features_to_select=16, step=1)
rfe_log.fit(X_train_cv, y_train_log)
X_train_log = X_train_cv[X_train_cv.columns[rfe_log.support_]]
X_test_log  = X_test_cv[X_train_log.columns]

lr_log = linear_model.LinearRegression()
lr_log.fit(X_train_log, y_train_log)

# Predict and convert back to original scale
y_pred_log  = np.expm1(lr_log.predict(X_test_log))
y_test_orig = y_test_cv

r2_log   = r2_score(y_test_orig, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test_orig, y_pred_log))

print('=== Log-Transform Model (evaluated on original scale) ===')
print(f'R²   : {r2_log:.3f}')
print(f'RMSE : {rmse_log:.2f}')


## Model Benchmarking

Comparing Linear Regression against Ridge (L2 regularisation), Lasso (L1), and Random Forest using 5-fold CV. This answers the question: *does regularisation help, and is linear regression the right choice for this problem?*

In [ ]:
X_train_bm, X_test_bm, y_train_bm, y_test_bm = get_model_ready(df)

# Use all features (let regularisation handle selection)
kf_bm = KFold(n_splits=5, shuffle=True, random_state=42)

benchmark_models = {
    'Linear Regression'   : linear_model.LinearRegression(),
    'Ridge  (alpha=1.0)'  : Ridge(alpha=1.0),
    'Lasso  (alpha=0.1)'  : Lasso(alpha=0.1),
    'Random Forest'       : RandomForestRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in benchmark_models.items():
    r2_scores   = cross_val_score(model, X_train_bm, y_train_bm, cv=kf_bm, scoring='r2')
    rmse_scores = np.sqrt(-cross_val_score(model, X_train_bm, y_train_bm, cv=kf_bm,
                                            scoring='neg_mean_squared_error'))
    results.append({
        'Model'        : name,
        'CV R² Mean'   : round(r2_scores.mean(), 3),
        'CV R² Std'    : round(r2_scores.std(),  3),
        'CV RMSE Mean' : round(rmse_scores.mean(), 1),
        'CV RMSE Std'  : round(rmse_scores.std(),  1),
    })

results_df = pd.DataFrame(results).sort_values('CV R² Mean', ascending=False)
print(results_df.to_string(index=False))


In [ ]:
# Visualise comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model_names = results_df['Model'].tolist()
r2_means    = results_df['CV R² Mean'].tolist()
r2_stds     = results_df['CV R² Std'].tolist()
rmse_means  = results_df['CV RMSE Mean'].tolist()
rmse_stds   = results_df['CV RMSE Std'].tolist()

x = range(len(model_names))

axes[0].bar(x, r2_means, yerr=r2_stds, capsize=5, color='steelblue', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0].set_title('CV R² by Model (higher = better)')
axes[0].set_ylabel('R²')

axes[1].bar(x, rmse_means, yerr=rmse_stds, capsize=5, color='salmon', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=15, ha='right')
axes[1].set_title('CV RMSE by Model (lower = better)')
axes[1].set_ylabel('RMSE')

plt.tight_layout()
plt.show()


### Benchmark Takeaways

- **Ridge / Lasso**: If R² is similar to plain Linear Regression, multicollinearity in the remaining features is not severe enough to require regularisation.
- **Random Forest**: A significantly higher R² here would indicate non-linear relationships that Linear Regression is missing — and would be a strong reason to switch models.
- **Choice justification**: The model selection decision should be documented in the README based on these numbers.

## Final Summary

The cell below consolidates all key metrics for easy comparison and README reporting.

In [ ]:
# Final held-out test evaluation for Linear Regression (RFE features)
X_train_f, X_test_f, y_train_f, y_test_f = get_model_ready(df)
rfe_f = RFE(linear_model.LinearRegression(), n_features_to_select=16, step=1)
rfe_f.fit(X_train_f, y_train_f)
X_tr_f = X_train_f[X_train_f.columns[rfe_f.support_]]
X_te_f = X_test_f[X_tr_f.columns]

lr_f = linear_model.LinearRegression()
lr_f.fit(X_tr_f, y_train_f)
y_pred_f = lr_f.predict(X_te_f)

n   = len(y_test_f)
p   = X_tr_f.shape[1]
r2  = r2_score(y_test_f, y_pred_f)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
rmse   = np.sqrt(mean_squared_error(y_test_f, y_pred_f))

summary = pd.DataFrame([
    {'Metric': 'R² (test)',          'Value': f'{r2:.3f}'},
    {'Metric': 'Adjusted R² (test)', 'Value': f'{adj_r2:.3f}'},
    {'Metric': 'RMSE (test)',         'Value': f'{rmse:.2f}'},
    {'Metric': 'CV R² (mean±std)',    'Value': f'{cv_r2.mean():.3f} ± {cv_r2.std():.3f}'},
    {'Metric': 'CV RMSE (mean±std)',  'Value': f'{cv_rmse.mean():.1f} ± {cv_rmse.std():.1f}'},
    {'Metric': 'Top features',        'Value': 'temp, yr, weathersit_3'},
])
print(summary.to_string(index=False))
